In [1]:
import numpy as np
import pandas as pd
from keras.layers import Dense, LSTM, Conv1D, Flatten, MaxPooling1D, Dropout
from keras.models import Sequential
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score 
import math
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from matplotlib import pyplot as plt
from IPython.core.pylabtools import figsize

In [2]:
data = pd.read_excel("sm.xls")

In [3]:
train_size = int(len(data)*0.7)
train_dataset, test_dataset = data.iloc[:train_size], data.iloc[train_size:]

In [4]:
# Split train data to X and y
X_train = train_dataset.drop('Eto', axis = 1)
y_train = train_dataset.loc[:,['Eto']]

# Split test data to X and y
X_test = test_dataset.drop('Eto', axis = 1)
y_test = test_dataset.loc[:,['Eto']]

In [5]:
scaler = MinMaxScaler()
X_train= scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [6]:
X_train_series = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_series = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

In [7]:
# Build the LSTM model
model = Sequential()
model.add(LSTM(32, return_sequences=True, input_shape=(5, 1)))
model.add(LSTM(16))
model.add(Dense(1))
model.compile(loss='mean_squared_error', optimizer='adam')
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 5, 32)             4352      
                                                                 
 lstm_1 (LSTM)               (None, 16)                3136      
                                                                 
 dense (Dense)               (None, 1)                 17        
                                                                 
Total params: 7,505
Trainable params: 7,505
Non-trainable params: 0
_________________________________________________________________


In [8]:
model.fit(X_train, y_train, validation_data=(X_test,y_test), batch_size=32,epochs=25, verbose=1)

Epoch 1/25
56/56 [==============================] - 9s 40ms/step - loss: 22.9088 - val_loss: 4.6936
Epoch 2/25
56/56 [==============================] - 1s 12ms/step - loss: 3.9343 - val_loss: 2.2551
Epoch 3/25
56/56 [==============================] - 1s 12ms/step - loss: 3.5393 - val_loss: 2.2102
Epoch 4/25
56/56 [==============================] - 1s 12ms/step - loss: 3.5339 - val_loss: 2.2317
Epoch 5/25
56/56 [==============================] - 1s 12ms/step - loss: 3.5274 - val_loss: 2.2301
Epoch 6/25
56/56 [==============================] - 1s 12ms/step - loss: 3.5231 - val_loss: 2.2357
Epoch 7/25
56/56 [==============================] - 1s 12ms/step - loss: 3.5152 - val_loss: 2.2075
Epoch 8/25
56/56 [==============================] - 1s 12ms/step - loss: 3.5074 - val_loss: 2.1714
Epoch 9/25
56/56 [==============================] - 1s 12ms/step - loss: 3.4980 - val_loss: 2.3043
Epoch 10/25
56/56 [==============================] - 1s 12ms/step - loss: 3.4966 - val_loss: 2.2395
Epoch 11

In [9]:
ypredtrain = model.predict(X_train)
print(model.evaluate(X_train, y_train))
print("MSE: %.4f" % mean_squared_error(y_train, ypredtrain))

56/56 [==============================] - 2s 4ms/step - loss: 0.1013
0.10125691443681717
MSE: 0.1013


In [10]:
# Evaluation metrices RMSE,MSE,  MAE and R square
print("Train data RMSE: ", math.sqrt(mean_squared_error(y_train,ypredtrain)))
print("Train data MSE: ", mean_squared_error(y_train,ypredtrain))
print("Train data MAE: ", mean_absolute_error(y_train,ypredtrain))
print("Train data R2 score:", r2_score(y_train , ypredtrain ))

Train data RMSE:  0.3182089443587127
Train data MSE:  0.10125693226988632
Train data MAE:  0.22784662984215792
Train data R2 score: 0.9718301061665043


In [11]:
## convert your array into a dataframe
df_pred = pd.DataFrame (ypredtrain)
df_pred.to_excel('Pred_LSTM_train__SM.xlsx')

In [13]:
ypredtest = model.predict(X_test)
print(model.evaluate(X_test, y_test))
print("MSE: %.4f" % mean_squared_error(y_test, ypredtest))

24/24 [==============================] - 0s 4ms/step - loss: 0.0522
0.05222158506512642
MSE: 0.0522


In [14]:
# Evaluation metrices RMSE, MSE, MAE and R square
print("Test data RMSE: ", math.sqrt(mean_squared_error(y_test,ypredtest)))
print("Test data MSE: ", mean_squared_error(y_test,ypredtest))
print("Test data MAE: ", mean_absolute_error(y_test,ypredtest))
print("Test data R2 score:", r2_score(y_test , ypredtest ))

Test data RMSE:  0.22852044495047527
Test data MSE:  0.05222159376036319
Test data MAE:  0.1824822788250943
Test data R2 score: 0.9758416586467812


In [15]:
## convert your array into a dataframe
df_pred = pd.DataFrame (ypredtest)
df_pred.to_excel('Pred_LSTM_test__SM.xlsx')